# Testing the GOESMultiCloud class functions, attributes and properties

In [2]:
import sys
sys.path.insert(0, '/Users/oscaryasunaga/Desktop/goesdatabuilder/goesdatabuilder')
from pathlib import Path
import os
import tempfile

In [3]:
from goesdatabuilder.data import GOESMultiCloud

In [4]:
import xarray as xr
import numpy as np
import netCDF4 as nc
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay

In [ ]:

FILE_PATH = "TEST"

In [ ]:

ds18 = xr.open_dataset(FILE_PATH, engine='netcdf4')

In [7]:
gmc2 = GOESMultiCloud

In [ ]:
gmc = GOESMultiCloud(FILE_PATH)

In [ ]:
assert ds18.attrs == gmc.attributes

In [ ]:
ds18.dataset_name  == gmc.dataset_name

True

In [ ]:
gmc.band = 3

In [ ]:
#TESTING PROPERTIES OF CMI_STATS

for i in range(1, 17):
    gmc.band = i
    band_name = f"CMI_C{str(i).zfill(2)}"

    # values check
    assert np.allclose(
        gmc.CMI["values"],
        ds18[band_name].values,
        equal_nan=True
    )

    # attributes check
    nc_attrs = ds18[band_name].attrs
    gmc_attrs = gmc.CMI["attributes"]

    for k, v in gmc_attrs.items():
        assert k in nc_attrs, f"Missing attr '{k}' in {band_name}"
        a = nc_attrs[k]
        b = v
        msg = f"Attribute '{k}' mismatch in {band_name}: expected {b}, got {a}"
        if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
            assert np.allclose(a, b, equal_nan=True), msg
        else: 
            assert nc_attrs[k] == v, f"{band_name} attr mismatch: {k}"

In [ ]:
#TESTING PROPERTIES OF DQF_STATS
for i in range(1, 17):
    gmc.band = i
    band_name = f"DQF_C{str(i).zfill(2)}"

    # values check
    assert np.allclose(
        gmc.DQF["values"],
        ds18[band_name].values,
        equal_nan=True
    )

    # attributes check
    nc_attrs = ds18[band_name].attrs
    gmc_attrs = gmc.DQF["attributes"]

    for k, v in gmc_attrs.items():
        assert k in nc_attrs, f"Missing attr '{k}' in {band_name}"
        a = nc_attrs[k]
        b = v
        msg = f"Attribute '{k}' mismatch in {band_name}: expected {b}, got {a}"
        if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
            assert np.allclose(a, b, equal_nan=True), msg
            
        else: 
            assert nc_attrs[k] == v, f"{band_name} attr mismatch: {k}"

In [131]:
#Satellite instruciton checks
assert gmc.orbital_slot == ds18.attrs['orbital_slot']
assert gmc.platform_id == ds18.attrs['platform_ID']
assert gmc.instrument_type == ds18.attrs['instrument_type']
assert gmc.instrument_id == ds18.attrs['instrument_ID']
assert gmc.scan_mode == ds18.attrs["timeline_id"]
assert gmc.scene_id == ds18.attrs["scene_id"]

In [ ]:
#Further checks for satellite projection


<xarray.DataArray 'goes_imager_projection' ()> Size: 4B
array(-2147483647, dtype=int32)
Coordinates:
    t        datetime64[ns] 8B ...
    y_image  float32 4B ...
    x_image  float32 4B ...
Attributes:
    long_name:                       GOES-R ABI fixed grid projection
    grid_mapping_name:               geostationary
    perspective_point_height:        35786023.0
    semi_major_axis:                 6378137.0
    semi_minor_axis:                 6356752.31414
    inverse_flattening:              298.2572221
    latitude_of_projection_origin:   0.0
    longitude_of_projection_origin:  -75.0
    sweep_angle_axis:                x

In [120]:
gmc.goes_imager_projection

{'values': array(-2147483647, dtype=int32),
 'attributes': {'long_name': 'GOES-R ABI fixed grid projection',
  'grid_mapping_name': 'geostationary',
  'perspective_point_height': np.float64(35786023.0),
  'semi_major_axis': np.float64(6378137.0),
  'semi_minor_axis': np.float64(6356752.31414),
  'inverse_flattening': np.float64(298.2572221),
  'latitude_of_projection_origin': np.float64(0.0),
  'longitude_of_projection_origin': np.float64(-75.0),
  'sweep_angle_axis': 'x'}}

In [143]:

gmc.time.values


np.datetime64('2018-01-03T02:05:57.928225984')

In [ ]:
np.allclose(gmc.time.values, ds18.coords.get('t').values, equal_nan=True)

np.True_

In [ ]:
gmc_y = gmc.y_axis
ds_y = ds18.coords.get('y')
np.allclose(gmc.y_axis.values, ds_y.values, equal_nan=True)

In [ ]:
gmc_x = gmc.x_axis
ds_x = ds18.coords.get('x')
np.allclose(gmc.x_axis.values, ds_x.values, equal_nan=True)


In [ ]:
gmc_x_center = gmc.x_center
ds_x_center = ds18.coords.get('x_image')

if ds_x_center is not None:
    assert np.allclose(gmc_x_center.values, ds_x_center.values, equal_nan=True)

In [ ]:
gmc_y_center = gmc.y_center
ds_y_center = ds18.coords.get('y_image')

if ds_y_center is not None:
    assert np.allclose(gmc_y_center.values, ds_y_center.values, equal_nan=True)

In [1]:
for band in range(1, 17):
    gmc.band = band
    coord_name = f'band_wavelength_C{band:02d}'

    gmc_wavelength = gmc.band_wavelength
    ds_wavelength = ds18.coords.get(coord_name)

    if ds_wavelength is not None:
        assert np.allclose(gmc_wavelength.values, ds_wavelength.values, equal_nan=True)


NameError: name 'gmc' is not defined

In [ ]:
for band in range(1, 17):
    gmc.band = band
    coord_name = f'band_id_C{band:02d}'

    gmc_band_id = gmc.band_id
    ds_band_id = ds18.coords.get(coord_name)

    if ds_band_id is not None:
        assert np.allclose(gmc_band_id.values, ds_band_id.values, equal_nan=True)


In [ ]:
assert set(gmc.variable_keys) == set(ds18.data_vars.keys())


In [ ]:
if 'time_bounds' in ds18.data_vars:
    gmc_time_bounds = gmc.time_bounds
    ds_time_bounds = ds18['time_bounds']

    assert np.allclose(
        gmc_time_bounds['values'],
        ds_time_bounds.values,
        equal_nan=True
    )

In [ ]:
if 'x_image_bounds' in ds18.data_vars:
    gmc_x_bounds = gmc.x_bounds
    ds_x_bounds = ds18['x_image_bounds']

    assert np.allclose(
        gmc_x_bounds['values'],
        ds_x_bounds.values,
        equal_nan=True
    )

In [ ]:
if 'y_image_bounds' in ds18.data_vars:
    gmc_y_bounds = gmc.y_bounds
    ds_y_bounds = ds18['y_image_bounds']

    assert np.allclose(
        gmc_y_bounds['values'],
        ds_y_bounds.values,
        equal_nan=True
    )

In [ ]:
gmc_proj = gmc.goes_imager_projection
ds_proj = ds18.data_vars.get('goes_imager_projection')

assert ds_proj is not None, "goes_imager_projection not found in dataset"

# Compare attributes
gmc_attrs = gmc_proj['attributes']
ds_attrs = dict(ds_proj.attrs)

for key, value in gmc_attrs.items():
    assert key in ds_attrs, f"Missing projection attribute '{key}'"

    ds_value = ds_attrs[key]

    if isinstance(value, (float, np.floating)):
        assert np.isclose(value, ds_value, equal_nan=True), \
            f"Projection attribute '{key}' mismatch"
    elif isinstance(value, np.ndarray):
        assert np.allclose(value, ds_value, equal_nan=True), \
            f"Projection attribute '{key}' mismatch"
    else:
        assert value == ds_value, \
            f"Projection attribute '{key}' mismatch"


In [ ]:
gmc_height = gmc.satellite_height
ds_height = ds18.data_vars.get('nominal_satellite_height')

assert ds_height is not None, "nominal_satellite_height not found in dataset"

assert np.allclose(
    gmc_height['values'],
    ds_height.values,
    equal_nan=True
)

In [ ]:
gmc_lon = gmc.subpoint_longitude
ds_lon = ds18.data_vars.get('nominal_satellite_subpoint_lon')

assert ds_lon is not None, "nominal_satellite_subpoint_lon not found in dataset"

assert np.allclose(
    gmc_lon['values'],
    ds_lon.values,
    equal_nan=True
)


In [ ]:
gmc_lat = gmc.subpoint_latitude
ds_lat = ds18.data_vars.get('nominal_satellite_subpoint_lat')

assert ds_lat is not None, "nominal_satellite_subpoint_lat not found in dataset"

assert np.allclose(
    gmc_lat['values'],
    ds_lat.values,
    equal_nan=True
)

In [ ]:
assert gmc.orbital_slot == ds18.attrs.get('orbital_slot')

In [ ]:
 assert gmc.spatial_resolution == ds18.attrs.get('spatial_resolution')

In [ ]:
assert gmc.production_site == ds18.attrs.get('production_site')

In [ ]:
        assert gmc.production_environment == ds.attrs.get('production_environment')


In [ ]:
mask = ~np.isnan(gmc.CMI["values"])
clean = gmc.CMI["values"][mask]
clean.mean()

gmc.CMI["values"]

In [133]:
FILE_PATH2 = "/Users/oscaryasunaga/Desktop/goesdatabuilder/goesdatabuilder/notebooks/data/OR_ABI-L2-MCMIPF-M3_G16_s20180030400393_e20180030411171_c20180030411245.nc"

gmc2 = GOESMultiCloud(FILE_PATH2)